# H100 full-pipeline smoke test

This notebook exercises two **independent** paths: (1) GPU-free NVIDIA Dynamo validation and optional AI Configurator sizing, then (2) native, single-node GPU benchmarks using deterministic grid sweeps and Optuna TPE. Dynamo generates reports/manifests; it does not deploy them or start inference servers. The benchmark cells launch and stop native servers one configuration at a time. Run the common setup cell, then either section on its own. Do not run GPU benchmark cells concurrently.

On the VM, use Python 3.11–3.13 and install this checkout with `pip install -e '.[optuna]'`. Install `wandb` and set `WANDB_PROJECT` (and optionally `WANDB_ENTITY`) to upload each native search and its log to W&B; local log files are saved even without W&B. Install the optional AI Configurator integration separately with `pip install -e '.[ai-configurator]'` if you will enable those cells. Install each backend in its own compatible environment/container; vLLM, SGLang, and TensorRT-LLM often have conflicting CUDA/PyTorch dependencies. Open this notebook with the corresponding environment's kernel for each framework. The example model is gated: accept its Hugging Face license and set `HF_TOKEN`, or change `MODEL` to a model all three installed backends support. TensorRT-LLM additionally needs a working `trtllm-serve` executable.

Outputs are deliberately disjoint: `results/notebook_h100/dynamo/` versus `results/notebook_h100/native/<framework>/<grid|optuna>/`. Grid and Optuna never append to each other's JSONL files or share an Optuna study. Existing results are only reused when `RESUME_EXISTING=True`; change `RUN_LABEL` for a fresh experiment.

In [1]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src/llm_optimizer/cli.py').exists():
            return candidate
    raise RuntimeError('Start Jupyter from the llm-optimizer checkout or a child directory.')

REPO = find_repo()
MODEL = "Qwen/Qwen3.5-4B"
RUN_LABEL = 'h100_smoke_v1'  # Change for a fresh run; keep unchanged to resume.
ROOT = REPO / 'results' / 'notebook_h100' / RUN_LABEL
DYNAMO_ROOT = ROOT / 'dynamo'
NATIVE_ROOT = ROOT / 'native'
RESUME_EXISTING = True

def run_cli(arguments):
    command = [sys.executable, '-m', 'llm_optimizer.cli', *map(str, arguments)]
    print('$', shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)

print('Repository:', REPO)
print('Model:', MODEL)
print('Output root:', ROOT)
print('Python:', sys.version.split()[0])

Repository: /workspace/llm-optimizer-trt
Model: Qwen/Qwen3.5-4B
Output root: /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1
Python: 3.12.3


In [2]:
import os
os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN"
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

## 1. Dynamo: static validation first

This section needs no local GPU or inference backend. It checks native parameters against the proposed topology and writes Kubernetes manifests into JSON reports. Defaults model one H100 worker (`TP=1`, one node, one GPU per node). If you increase the topology, update the TP/PP/CP values so their product equals `NODE_COUNT × GPUS_PER_NODE`. Keep `CHECK_CLUSTER=False` unless a compatible Kubernetes API and Dynamo CRD are available; server-side dry-run does not deploy resources.

In [12]:
NODE_COUNT = 1
GPUS_PER_NODE = 1
DYNAMO_GPU_SYSTEM = 'h100_sxm'  # Analytical model; not a local GPU allocation.
CHECK_CLUSTER = False
DYNAMO_SERVER_ARGS = {
    'vllm': 'tensor_parallel_size=1;pipeline_parallel_size=1;max_num_seqs=[16,32];max_num_batched_tokens=8192',
    'sglang': 'tp_size=1;pp_size=1;max_running_requests=[16,32];max_total_tokens=8192',
    'trtllm': 'tp_size=1;pp_size=1;cp_size=1;max_batch_size=[16,32];max_num_tokens=8192',
}

def dynamo_check(framework):
    output = DYNAMO_ROOT / 'static' / f'{framework}.json'
    output.parent.mkdir(parents=True, exist_ok=True)
    args = ['dynamo-check', '--model', MODEL, '--framework', framework,
            '--node-count', NODE_COUNT, '--gpus-per-node', GPUS_PER_NODE,
            '--server-args', DYNAMO_SERVER_ARGS[framework], '--output', output]
    if CHECK_CLUSTER:
        args.append('--check-cluster')
    run_cli(args)
    report = json.loads(output.read_text())
    assert report['valid'], report
    print(framework, 'validated configurations:', len(report['configurations']))
    return report

assert NODE_COUNT > 0 and GPUS_PER_NODE > 0

In [13]:
dynamo_vllm = dynamo_check('vllm')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-check --model Qwen/Qwen3.5-4B --framework vllm --node-count 1 --gpus-per-node 1 --server-args 'tensor_parallel_size=1;pipeline_parallel_size=1;max_num_seqs=[16,32];max_num_batched_tokens=8192' --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/vllm.json
Wrote Dynamo report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/vllm.json
vllm validated configurations: 2


In [5]:
dynamo_sglang = dynamo_check('sglang')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-check --model meta-llama/Llama-3.1-8B-Instruct --framework sglang --node-count 1 --gpus-per-node 1 --server-args 'tp_size=1;pp_size=1;max_running_requests=[16,32];max_total_tokens=8192' --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/sglang.json
Wrote Dynamo report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/sglang.json
sglang validated configurations: 2


In [6]:
dynamo_trtllm = dynamo_check('trtllm')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-check --model meta-llama/Llama-3.1-8B-Instruct --framework trtllm --node-count 1 --gpus-per-node 1 --server-args 'tp_size=1;pp_size=1;cp_size=1;max_batch_size=[16,32];max_num_tokens=8192' --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/trtllm.json
Wrote Dynamo report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/static/trtllm.json
trtllm validated configurations: 2


### Optional: AI Configurator GPU-free sizing

Enable only if `aiconfigurator` is installed in this kernel's environment. This runs a separate topology-sizing workflow for each framework; it does not consume static-check outputs or native benchmark results. Its H100 GPU-system value is a model input, not a request to reserve the VM GPU. Generated manifests and ranked CSVs stay under `dynamo/aiconfigurator/`.

In [3]:
RUN_AICONFIGURATOR = True

def dynamo_optimize(framework):
    if not RUN_AICONFIGURATOR:
        print('Skipped AI Configurator for', framework)
        return None
    output = DYNAMO_ROOT / 'aiconfigurator' / f'{framework}.json'
    artifacts = DYNAMO_ROOT / 'aiconfigurator' / framework
    output.parent.mkdir(parents=True, exist_ok=True)
    args = ['dynamo-optimize', '--model', MODEL, '--framework', framework,
            '--node-count', NODE_COUNT, '--gpus-per-node', GPUS_PER_NODE,
            '--gpu-system', DYNAMO_GPU_SYSTEM, '--input-len', 512,
            '--output-len', 128, '--ttft', 2000, '--tpot', 100,
            '--artifacts-dir', artifacts, '--output', output]
    if CHECK_CLUSTER:
        args.append('--check-cluster')
    run_cli(args)
    report = json.loads(output.read_text())
    assert report['valid'], report
    return report


In [9]:
%pip install "aisimulate==0.1.0.dev2"

  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pyarrow-25.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached protobuf-7.36.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
INFO: pip is looking at multiple versions of orbax-checkpoint to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of orbax-checkpoint to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 MB 31.4 MB/s  0:00:05m0:00:0100:01
   ━━━━

In [15]:
aic_vllm = dynamo_optimize('vllm')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-optimize --model Qwen/Qwen3.5-4B --framework vllm --node-count 1 --gpus-per-node 1 --gpu-system h100_sxm --input-len 512 --output-len 128 --ttft 2000 --tpot 100 --artifacts-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/vllm --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/vllm.json
Wrote AI Configurator report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/vllm.json


In [16]:
aic_sglang = dynamo_optimize('sglang')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-optimize --model Qwen/Qwen3.5-4B --framework sglang --node-count 1 --gpus-per-node 1 --gpu-system h100_sxm --input-len 512 --output-len 128 --ttft 2000 --tpot 100 --artifacts-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/sglang --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/sglang.json
Wrote AI Configurator report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/sglang.json


In [17]:
aic_trtllm = dynamo_optimize('trtllm')

$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli dynamo-optimize --model Qwen/Qwen3.5-4B --framework trtllm --node-count 1 --gpus-per-node 1 --gpu-system h100_sxm --input-len 512 --output-len 128 --ttft 2000 --tpot 100 --artifacts-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/trtllm --output /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/trtllm.json
Wrote AI Configurator report to /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/dynamo/aiconfigurator/trtllm.json


## 2. Native GPU parameter optimization

This section depends only on the **common setup**, not any Dynamo cell. Run it in a backend-capable VM environment with an idle H100 (or adjust `GPU_COUNT` and parameters). The CLI starts its own native server and terminates it between configurations. Grid and Optuna use distinct outputs even for the same framework. Each native search saves the combined framework-server and llm-optimizer console stream to its own `search.log`, including startup failures and messages between trials; when W&B is configured, the log is uploaded to that search's W&B run. The fixed synthetic workload (seed, prompt count, input/output lengths) makes sweeps comparable; performance timings will still vary. Set either run switch to `True` after reviewing the printed plans. Each grid currently has 8 combinations; each Optuna space has 18 with a 12-unique-configuration budget. These are real model reloads and can take substantial time.

In [3]:
#### On H200 ####
GPU_COUNT = 1
RUN_GRID = True
RUN_OPTUNA = True
WANDB_PROJECT = "full_llm_optimize_pipeline"  # Leave unset for local logs only.
WANDB_ENTITY = "maorblumberg-tel-aviv-university"
OPTUNA_TRIALS = 12
OPTUNA_TIMEOUT_SECONDS = 7200
REST_SECONDS = 5
OBJECTIVE = 'output_throughput'
SLO = 'ttft:p95<2000ms;itl:p95<100ms'
CLIENT_GRID = 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16'
CLIENT_OPTUNA = CLIENT_GRID
SERVER_GRID = {
    'vllm': 'tensor_parallel_size=1;max_num_seqs=[16,32];max_num_batched_tokens=[4096,8192]',
    'sglang': 'tp_size=1;max_running_requests=[16,32];max_total_tokens=[4096,8192]',
    'trtllm': 'tp_size=1;max_batch_size=[16,32];max_num_tokens=[4096,8192]',
}
SERVER_OPTUNA = {
    'vllm': (
        'tensor_parallel_size=1;'
        'max_num_seqs=[8,16,24,32,48,64,96];'
        'max_num_batched_tokens=[2048,4096,6144,8192,12288,16384]'
    ),
    'sglang': (
        'tp_size=1;'
        'max_running_requests=[8,16,24,32,48,64,96];'
        'max_total_tokens=[2048,4096,6144,8192,12288,16384]'
    ),
    'trtllm': (
        'tp_size=1;'
        'max_batch_size=[8,16,24,32,48,64,96];'
        'max_num_tokens=[2048,4096,6144,8192,12288,16384]'
    ),
}
PORTS = {'vllm': 8000, 'sglang': 30000, 'trtllm': 8001}

try:
    gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                               text=True, capture_output=True, check=False)
except FileNotFoundError:
    gpu_probe = None
if gpu_probe is not None and gpu_probe.returncode == 0:
    detected = [line.strip() for line in gpu_probe.stdout.splitlines() if line.strip()]
    print('Visible GPUs:', detected)
    assert len(detected) >= GPU_COUNT, 'Not enough visible GPUs for native sweeps'
    if not all('H100' in name for name in detected[:GPU_COUNT]):
        print('Warning: this is not an H100 run; compare results accordingly.')
else:
    print('nvidia-smi unavailable; verify the VM GPU before enabling sweeps.')


Visible GPUs: ['NVIDIA H200']


In [4]:
import hashlib
import signal
from datetime import datetime, timezone

def run_logged_cli(arguments, log_path):
    command = [sys.executable, '-m', 'llm_optimizer.cli', *map(str, arguments)]
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        header = f'\n=== {datetime.now(timezone.utc).isoformat()} $ {shlex.join(command)} ===\n'
        log.write(header)
        print(header, end='', flush=True)
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                print(line, end='', flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
                process.wait(timeout=10)
            raise
        finally:
            if process.stdout is not None:
                process.stdout.close()
        log.write(f'=== exit code: {return_code} ===\n')
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)

def benchmark_plan(framework, strategy):
    output_dir = NATIVE_ROOT / framework / strategy
    output = output_dir / 'results.json'
    server_args = SERVER_GRID[framework] if strategy == 'grid' else SERVER_OPTUNA[framework]
    client_args = CLIENT_GRID if strategy == 'grid' else CLIENT_OPTUNA
    args = ['--framework', framework, '--model', MODEL, '--gpus', GPU_COUNT,
            '--host', '127.0.0.1', '--port', PORTS[framework],
            '--server-args', server_args, '--client-args', client_args,
            '--constraints', SLO, '--rest', REST_SECONDS,
            '--output-dir', output_dir, '--output-json', output]
    if strategy == 'optuna':
        args += ['--search-strategy', 'optuna', '--optuna-trials', OPTUNA_TRIALS,
                 '--optuna-timeout', OPTUNA_TIMEOUT_SECONDS, '--optuna-objective', OBJECTIVE,
                 '--optuna-seed', 1, '--optuna-study-name', f'{RUN_LABEL}-{framework}-native']
    return output, args

def native_run(framework, strategy):
    output, args = benchmark_plan(framework, strategy)
    enabled = RUN_GRID if strategy == 'grid' else RUN_OPTUNA
    print(framework, strategy, 'output:', output)
    if not enabled:
        print('Preview only; set RUN_GRID or RUN_OPTUNA to True to execute.')
        print('$', shlex.join([sys.executable, '-m', 'llm_optimizer.cli', *map(str, args), '--dry-run']))
        return None
    jsonl = output.with_suffix('.jsonl')
    database = output.with_suffix('.optuna.db')
    has_state = output.exists() or jsonl.exists() or (strategy == 'optuna' and database.exists())
    if has_state and not RESUME_EXISTING:
        raise FileExistsError(f'Existing {strategy} run at {output}; change RUN_LABEL or enable resume')
    if has_state:
        args.append('--continue')
    output.parent.mkdir(parents=True, exist_ok=True)
    run_cli([*args, '--dry-run'])
    search_log = output.parent / 'search.log'
    wb_run = None
    if WANDB_PROJECT:
        import wandb
        run_key = f'{RUN_LABEL}:{framework}:{strategy}:{MODEL}'
        run_id = hashlib.sha256(run_key.encode()).hexdigest()[:16]
        wb_run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, id=run_id,
                            resume='allow', name=f'{RUN_LABEL}-{framework}-{strategy}_2',
                            group=f'{RUN_LABEL}-{framework}-{strategy}', job_type=strategy,
                            config={'model': MODEL, 'framework': framework, 'strategy': strategy,
                                    'server_args': SERVER_GRID[framework] if strategy == 'grid' else SERVER_OPTUNA[framework],
                                    'client_args': CLIENT_GRID if strategy == 'grid' else CLIENT_OPTUNA,
                                    'gpus': GPU_COUNT, 'objective': OBJECTIVE})
    try:
        run_logged_cli(args, search_log)
        if not output.exists():
            raise RuntimeError(f'No aggregate result generated: {output}')
        report = json.loads(output.read_text())
        print('Successful benchmarks:', report['metadata']['total_tests'])
        if strategy == 'optuna':
            print('Optuna summary:', report['optimization'])
        if wb_run is not None:
            best = report['best_configurations']['best_output_throughput']
            wb_run.log({'successful_benchmarks': report['metadata']['total_tests'],
                        'best_output_throughput': None if best is None else best['throughput']})
        return report
    finally:
        if wb_run is not None:
            if search_log.exists():
                artifact = wandb.Artifact(f'{RUN_LABEL}-{framework}-{strategy}-log', type='logs')
                artifact.add_file(str(search_log))
                wb_run.log_artifact(artifact)
            wb_run.finish()

assert DYNAMO_ROOT != NATIVE_ROOT
for framework in PORTS:
    assert benchmark_plan(framework, 'grid')[0] != benchmark_plan(framework, 'optuna')[0]

### vLLM: grid, then Optuna

Run these independently of Dynamo and of the other frameworks. The first cell is a deterministic 8-configuration sweep; the second is a 12-trial Optuna search in an 18-configuration space.

In [7]:
vllm_grid = native_run('vllm', 'grid')

vllm grid output: /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/grid/results.json
$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli --framework vllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8000 --server-args 'tensor_parallel_size=1;max_num_seqs=[16,32];max_num_batched_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=[8,16]' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/grid --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/grid/results.json --dry-run
2026-09-16 20:29:39,823 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 20:29:39,841 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
Dry run mode enabled.
2026-09-16 20:29:39

2026-09-16 20:29:39,823 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 20:29:39,841 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 20:29:39,842 [INFO] Generated 8 configuration(s) to run.
2026-09-16 20:29:39,842 [INFO] --------------------------------------------------------------------------------
2026-09-16 20:29:39,842 [INFO] Starting run 1/8: client_dataset_name-random-max_concurrency-8-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_num_batched_tokens-4096-max_num_seqs-16-tensor_parallel_size-1
2026-09-16 20:29:39,842 [INFO] --------------------------------------------------------------------------------
2026-09-16 20:29:39,843 [INFO] Starting run 2/8: client_dataset_name-random-max_concurrency-16-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_num_batched_tokens-4096-max_num_seqs-16-tensor_parallel_size-1
2026-09-16 20:29:39,843 [INFO] ----------------------------------------------------------


=== 2026-09-16T20:29:43.571420+00:00 $ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli --framework vllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8000 --server-args 'tensor_parallel_size=1;max_num_seqs=[16,32];max_num_batched_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=[8,16]' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/grid --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/grid/results.json ===
2026-09-16 20:29:43,855 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 20:29:43,855 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 20:29:43,872 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
2026-09-16 20:29:43,872 [INFO] Detected

best_output_throughput,▁
successful_benchmarks,▁
best_output_throughput,1364.3951
successful_benchmarks,8


In [14]:
vllm_optuna = native_run('vllm', 'optuna')

vllm optuna output: /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/optuna/results.json
$ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli --framework vllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8000 --server-args 'tensor_parallel_size=1;max_num_seqs=[8,16,24,32,48,64,96];max_num_batched_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/optuna --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-vllm-native --dry-run
2026-09-16 20:51:46,178 - llm_op

2026-09-16 20:51:46,178 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 20:51:46,197 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 20:51:46,198 [INFO] Generated 42 configuration(s) to run.



=== 2026-09-16T20:51:47.523570+00:00 $ /home/ubuntu/llm-optimizer-trt/.venv-vllm/bin/python -m llm_optimizer.cli --framework vllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8000 --server-args 'tensor_parallel_size=1;max_num_seqs=[8,16,24,32,48,64,96];max_num_batched_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/optuna --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/vllm/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-vllm-native ===
2026-09-16 20:51:47,814 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 

best_output_throughput,▁
successful_benchmarks,▁
best_output_throughput,1391.16314
successful_benchmarks,12


### SGLang: grid, then Optuna

Uses native SGLang option names and the same synthetic client workload. Ensure no vLLM server is still running before enabling these cells.

In [6]:
sglang_grid = native_run('sglang', 'grid')

sglang grid output: /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/grid/results.json
$ /home/ubuntu/llm-optimizer-trt/.venv-sglang/bin/python -m llm_optimizer.cli --framework sglang --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 30000 --server-args 'tp_size=1;max_running_requests=[16,32];max_total_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/grid --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/grid/results.json --dry-run
2026-09-16 21:28:02,720 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 21:28:02,739 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
Dry run mode enabled.
2026-09-16 21:28:02,7

2026-09-16 21:28:02,720 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 21:28:02,739 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 21:28:02,741 [INFO] Generated 4 configuration(s) to run.
2026-09-16 21:28:02,741 [INFO] --------------------------------------------------------------------------------
2026-09-16 21:28:02,741 [INFO] Starting run 1/4: client_dataset_name-random-max_concurrency-16-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_running_requests-16-max_total_tokens-4096-tp_size-1
2026-09-16 21:28:02,741 [INFO] --------------------------------------------------------------------------------
2026-09-16 21:28:02,741 [INFO] Starting run 2/4: client_dataset_name-random-max_concurrency-16-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_running_requests-16-max_total_tokens-8192-tp_size-1
2026-09-16 21:28:02,741 [INFO] -------------------------------------------------------------------------------


=== 2026-09-16T21:28:06.558481+00:00 $ /home/ubuntu/llm-optimizer-trt/.venv-sglang/bin/python -m llm_optimizer.cli --framework sglang --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 30000 --server-args 'tp_size=1;max_running_requests=[16,32];max_total_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/grid --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/grid/results.json ===
2026-09-16 21:28:06,855 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 21:28:06,855 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 21:28:06,873 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
2026-09-16 21:28:06,873 [INFO] Detected GPU t

best_output_throughput,▁
successful_benchmarks,▁
best_output_throughput,1615.77832
successful_benchmarks,4


In [ ]:
sglang_optuna = native_run('sglang', 'optuna')

sglang optuna output: /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/optuna/results.json
$ /home/ubuntu/llm-optimizer-trt/.venv-sglang/bin/python -m llm_optimizer.cli --framework sglang --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 30000 --server-args 'tp_size=1;max_running_requests=[8,16,24,32,48,64,96];max_total_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/optuna --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-sglang-native --dry-run
2026-09-16 21:40:28,943 - ll

2026-09-16 21:40:28,943 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 21:40:28,962 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 21:40:28,963 [INFO] Generated 42 configuration(s) to run.



=== 2026-09-16T21:40:31.750191+00:00 $ /home/ubuntu/llm-optimizer-trt/.venv-sglang/bin/python -m llm_optimizer.cli --framework sglang --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 30000 --server-args 'tp_size=1;max_running_requests=[8,16,24,32,48,64,96];max_total_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/optuna --output-json /home/ubuntu/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/sglang/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-sglang-native ===
2026-09-16 21:40:32,054 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 

### TensorRT-LLM: grid, then Optuna

Uses `trtllm-serve` and native underscore-style options. Verify the installed TensorRT-LLM release supports this model and these CLI arguments before enabling the run switches.

In [5]:
trtllm_grid = native_run('trtllm', 'grid')

trtllm grid output: /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/grid/results.json
$ /usr/bin/python3 -m llm_optimizer.cli --framework trtllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8001 --server-args 'tp_size=1;max_batch_size=[16,32];max_num_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/grid --output-json /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/grid/results.json --dry-run
2026-09-16 22:21:04,379 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:21:04,397 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
Dry run mode enabled.
2026-09-16 22:21:04,398 - llm_optimizer.main - INFO - Generated 4 configur

2026-09-16 22:21:04,379 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:21:04,397 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 22:21:04,398 [INFO] Generated 4 configuration(s) to run.
2026-09-16 22:21:04,399 [INFO] --------------------------------------------------------------------------------
2026-09-16 22:21:04,399 [INFO] Starting run 1/4: client_dataset_name-random-max_concurrency-16-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_batch_size-16-max_num_tokens-4096-tp_size-1
2026-09-16 22:21:04,399 [INFO] --------------------------------------------------------------------------------
2026-09-16 22:21:04,399 [INFO] Starting run 2/4: client_dataset_name-random-max_concurrency-16-num_prompts-128-random_input_len-512-random_output_len-128-seed-1_server_max_batch_size-16-max_num_tokens-8192-tp_size-1
2026-09-16 22:21:04,399 [INFO] --------------------------------------------------------------------------------
2026-09-16 22:


=== 2026-09-16T22:21:08.089714+00:00 $ /usr/bin/python3 -m llm_optimizer.cli --framework trtllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8001 --server-args 'tp_size=1;max_batch_size=[16,32];max_num_tokens=[4096,8192]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/grid --output-json /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/grid/results.json ===
2026-09-16 22:21:08,401 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:21:08,401 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:21:08,419 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
2026-09-16 22:21:08,419 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 22:21:08,421 - llm_o

best_output_throughput,▁
successful_benchmarks,▁
best_output_throughput,1260.62979
successful_benchmarks,4


In [6]:
trtllm_optuna = native_run('trtllm', 'optuna')

trtllm optuna output: /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/optuna/results.json
$ /usr/bin/python3 -m llm_optimizer.cli --framework trtllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8001 --server-args 'tp_size=1;max_batch_size=[8,16,24,32,48,64,96];max_num_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/optuna --output-json /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-trtllm-native --dry-run


2026-09-16 22:32:48,924 [INFO] Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:32:48,943 [INFO] Detected GPU type: H200, Count: 1
2026-09-16 22:32:48,945 [INFO] Generated 42 configuration(s) to run.


2026-09-16 22:32:48,924 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:32:48,943 - llm_optimizer.main - INFO - Detected GPU type: H200, Count: 1
2026-09-16 22:32:48,945 - llm_optimizer.main - INFO - Generated 42 configuration(s) to run.
Optuna dry run mode enabled.
Framework: trtllm
Search-space size: 42
Objective: output_throughput (maximize)
Trial budget: 12
Timeout: 7200.0
Storage: sqlite
Study name: h100_smoke_v1-trtllm-native



=== 2026-09-16T22:32:50.331043+00:00 $ /usr/bin/python3 -m llm_optimizer.cli --framework trtllm --model Qwen/Qwen3.5-4B --gpus 1 --host 127.0.0.1 --port 8001 --server-args 'tp_size=1;max_batch_size=[8,16,24,32,48,64,96];max_num_tokens=[2048,4096,6144,8192,12288,16384]' --client-args 'dataset_name=random;random_input_len=512;random_output_len=128;num_prompts=128;seed=1;max_concurrency=16' --constraints 'ttft:p95<2000ms;itl:p95<100ms' --rest 5 --output-dir /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/optuna --output-json /workspace/llm-optimizer-trt/results/notebook_h100/h100_smoke_v1/native/trtllm/optuna/results.json --search-strategy optuna --optuna-trials 12 --optuna-timeout 7200 --optuna-objective output_throughput --optuna-seed 1 --optuna-study-name h100_smoke_v1-trtllm-native ===
2026-09-16 22:32:50,631 - llm_optimizer.main - INFO - Parsed 2 constraint(s): ttft:p95<2000ms;itl:p95<100ms
2026-09-16 22:32:50,631 [INFO] Parsed 2 constraint(s): ttft:p9

best_output_throughput,▁
successful_benchmarks,▁
best_output_throughput,1336.85941
successful_benchmarks,12


## Inspect saved results

This cell reads whichever native reports exist; it does not start servers. Compare throughput only among runs with matching model, workload, GPU, and backend versions. Dynamo sizing outputs describe deployments, not these direct benchmark timings.

In [7]:
for framework in ('vllm', 'sglang', 'trtllm'):
    for strategy in ('grid', 'optuna'):
        output, _ = benchmark_plan(framework, strategy)
        if not output.exists():
            continue
        report = json.loads(output.read_text())
        best = report['best_configurations']['best_output_throughput']
        feasible = report['best_configurations']['best_output_throughput_constrained']
        print(framework, strategy, 'successes=', report['metadata']['total_tests'],
              'best_output_tok_s=', None if best is None else best['throughput'],
              'best_feasible_tok_s=', None if feasible is None else feasible['throughput'])
        if strategy == 'optuna':
            print('  states=', report['optimization']['trial_state_counts'],
                  'stop=', report['optimization']['stop_reason'])

vllm grid successes= 8 best_output_tok_s= 1364.3950979384279 best_feasible_tok_s= 1364.3950979384279
vllm optuna successes= 12 best_output_tok_s= 1391.163142484229 best_feasible_tok_s= 1391.163142484229
  states= {'COMPLETE': 12} stop= trial_budget
sglang grid successes= 4 best_output_tok_s= 1615.7783222727996 best_feasible_tok_s= 1615.7783222727996
sglang optuna successes= 12 best_output_tok_s= 1673.60075069069 best_feasible_tok_s= 1673.60075069069
  states= {'COMPLETE': 12, 'PRUNED': 1} stop= trial_budget
trtllm grid successes= 4 best_output_tok_s= 1260.629786466157 best_feasible_tok_s= 1260.629786466157
trtllm optuna successes= 12 best_output_tok_s= 1336.8594090952583 best_feasible_tok_s= 1336.8594090952583
  states= {'COMPLETE': 12} stop= trial_budget
